# Fluidic pinball: LatentNP vs AttentiveNP

Trains both models on the pinball data with the same data, settings and seed, and records for each:
**number of parameters**, **training time**, and the validation and test metrics.
Results are saved to `OUTPUT_DIR/results.json`, with checkpoints in one folder per model.

Data (`pinball_data.py`): each task is one trajectory at one time t0, with the fixed sensors' recent
readings as context and 2048–4096 other random mesh nodes at t0 as targets.
- `x = ([mu,] relative time, x, y)`; the model maps its domain to [-1, 1] internally.
- `y` is standardized with the training mean and std (saved with the results).
- Training follows the previous scripts: phase 1 with KL weight 0, then phase 2 with the KL weight
  ramping to 1, a plateau scheduler and early stopping.
- The data is small, so it lives on the GPU and batches are built there (no loader workers).
- Validation uses a fixed set of tasks, drawn once, every `VALIDATE_EVERY` epochs.
  Test: every mesh node at t0 = 30, for lags 0, 9, 19.

## Setup

In [ ]:
import json
import math
import os
import sys
import time
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# The folder with base.py, layers.py, LNP.py, ANP.py, loss.py, training.py, pinball_data.py:
# by default this notebook's own folder (a batch job can set it with NP_CODE_DIR).
MODEL_CODE_DIR = Path(os.environ.get("NP_CODE_DIR", ".")).resolve()
if str(MODEL_CODE_DIR) in sys.path:
    sys.path.remove(str(MODEL_CODE_DIR))
sys.path.insert(0, str(MODEL_CODE_DIR))  # first, so `import LNP` finds LNP.py

# An old LNP/ package in the same folder as LNP.py would be imported instead of it: catch that.
import importlib.util
_spec = importlib.util.find_spec("LNP")
if _spec is None or not (_spec.origin or "").endswith("LNP.py"):
    _found = _spec and (_spec.origin or list(_spec.submodule_search_locations or []))
    raise ImportError(f"`import LNP` finds {_found}, not LNP.py in {MODEL_CODE_DIR}. "
                      "Put the new model files in their own folder, or rename the old LNP/ package.")

from base import NPConfig
from LNP import LatentNP
from ANP import AttentiveNP
from training import train
from pinball_data import TrainBatches, load_pinball, make_test_batch, make_validation_batches, x_bounds

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}" + ("" if device.type == "cuda" else "  (no GPU: training will be very slow)"))
if device.type == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name()}, {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB")

## Settings
The same values as the previous `pinball_2.py` (ANP) / `pinball_3.py` (LNP) scripts.

In [ ]:
# Paths: a batch job can override them with PINBALL_DATA_DIR and PINBALL_OUTPUT_DIR.
DATA_DIR = Path(os.environ.get("PINBALL_DATA_DIR", "."))  # Pinball_mesh.xml, Pinball_data.npz
OUTPUT_DIR = Path(os.environ.get("PINBALL_OUTPUT_DIR", "results_pinball_comparison"))

USE_MU = False                        # True: the 3 flow parameters are added to x
SENSORS = [1573, 6925, 1986]          # mesh node indices of the fixed sensors
BATCH_SIZE = 16
VALIDATION_ROUNDS = 4                 # fixed validation set: 4 passes over the validation trajectories
VALIDATE_EVERY = 10                   # epochs between validations (an epoch is only 25 steps)
SEED = 0

MODEL_SETTINGS = dict(
    r_dim=128, z_dim=128, hidden_dim=128, n_hidden=2, dropout=0.05, activation=nn.ReLU, norm=True,
    use_fourier=True, fourier_dims=3, num_frequencies=32, fourier_scale=1.0, learnable_fourier=True,
    n_decoder_hidden=3, num_heads=4,
)

# Phase 2: KL weight ramps 0 -> 1, then plateau scheduler and early stopping.
EPOCHS, LR = 500, 1e-3
KL_RAMP_EPOCHS = 50                 # also: no LR plateau steps or early stopping before this
PLATEAU_PATIENCE, EARLY_STOPPING_PATIENCE = 32, 64   # both in epochs
GRAD_CLIP = 1.0
PRINT_EVERY = 10

RUNS = {"LatentNP": LatentNP, "AttentiveNP": AttentiveNP}

TEST_TIME, TEST_LAGS, TEST_MC_SAMPLES = 30, [0, 9, 19], 100

## Data

In [ ]:
data = load_pinball(DATA_DIR, seed=SEED).to(device)   # the whole dataset on the GPU
for split, y in data.y.items():
    print(f"{split}: {tuple(y.shape)}  (trajectories, times, mesh nodes)")
print(f"y: mean {data.y_mean:.4g}, std {data.y_std:.4g} (training set)")
print(f"Sensors: {SENSORS} at {data.coordinates[SENSORS].tolist()}")

x_low, x_high = x_bounds(data, USE_MU)
print(f"x domain: low {[round(v, 4) for v in x_low]}, high {[round(v, 4) for v in x_high]}")

train_batches = TrainBatches(data, SENSORS, USE_MU, batch_size=BATCH_SIZE)
val_batches = make_validation_batches(data, SENSORS, USE_MU, n_rounds=VALIDATION_ROUNDS,
                                      batch_size=BATCH_SIZE, seed=SEED)
batch = val_batches[0]
print(f"{len(train_batches)} training batches per epoch, {len(val_batches)} validation batches")
print("Batch:", {key: tuple(value.shape) for key, value in batch.items()})
X_DIM, Y_DIM = batch["x_context"].shape[-1], batch["y_context"].shape[-1]

## Training

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def format_duration(seconds):
    hours, rest = divmod(int(round(seconds)), 3600)
    return f"{hours}:{rest // 60:02d}:{rest % 60:02d}"


def make_config():
    # Targets and context are separate sets: the model joins them for the posterior.
    return NPConfig(x_dim=X_DIM, y_dim=Y_DIM, x_low=x_low, x_high=x_high,
                    target_includes_context=False, **MODEL_SETTINGS)


def save_results():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    payload = {
        "results": results,
        "normalization": {"y_mean": data.y_mean, "y_std": data.y_std},
        "settings": {"use_mu": USE_MU, "sensors": SENSORS, "batch_size": BATCH_SIZE,
                     "validate_every": VALIDATE_EVERY,
                     "phase2": {"epochs": EPOCHS, "lr": LR, "kl_ramp_epochs": KL_RAMP_EPOCHS,
                                "plateau_patience": PLATEAU_PATIENCE,
                                "early_stopping_patience": EARLY_STOPPING_PATIENCE},
                     "seed": SEED, "device": str(device)},
    }
    with open(OUTPUT_DIR / "results.json", "w") as f:
        json.dump(payload, f, indent=2, default=str)


def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()


results, histories, models = {}, {}, {}

for name, model_class in RUNS.items():
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
    torch.manual_seed(SEED)
    model = model_class(make_config()).to(device)
    n_params = count_parameters(model)
    print(f"Parameters: {n_params:,}")

    synchronize()
    start = time.perf_counter()

    print(f"--- Training: up to {EPOCHS} epochs, KL weight 0 -> 1 over {KL_RAMP_EPOCHS}")
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    # The scheduler is stepped at each validation, so its patience is counted in validations.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=max(1, PLATEAU_PATIENCE // VALIDATE_EVERY), min_lr=1e-6)
    history = train(model, train_batches, optimizer, EPOCHS, device, val_loader=val_batches,
                      validate_every=VALIDATE_EVERY, scheduler=scheduler, warmup_epochs=KL_RAMP_EPOCHS,
                      kl_schedule=[e / KL_RAMP_EPOCHS for e in range(KL_RAMP_EPOCHS)] + [1.0],
                      grad_clip=GRAD_CLIP, early_stopping_patience=EARLY_STOPPING_PATIENCE,
                      checkpoint_dir=OUTPUT_DIR / name / "phase2", print_every=PRINT_EVERY, monitor = "elbo")

    synchronize()
    seconds = time.perf_counter() - start

    epochs = len(history["train_loss"])
    history["val_epoch"] = [e for e in history["val_epoch"]]
    # best validation of phase 2 after the KL ramp (the model was restored to it)
    best_elbo, best_nll, best_mse, best_epoch = min(
        (el, n, m, e) for el, n, m, e in zip(history["val_elbo"], history["val_nll"],
                                            history["val_mse"], history["val_epoch"])
        if e > KL_RAMP_EPOCHS or e == history["val_epoch"][-1])
    results[name] = {
        "parameters": n_params,
        "training_time_s": round(seconds, 1),
        "training_time": format_duration(seconds),
        "seconds_per_epoch": round(seconds / epochs, 2),
        "epochs_run": epochs,
        "best_epoch": best_epoch,
        "best_val_elbo": best_elbo,
        "best_val_nll": best_nll,
        "best_val_mse": best_mse,
        "config": asdict(model.cfg),
    }
    histories[name], models[name] = history, model
    save_results()  # after each model, so a crash in the second run keeps the first
    print(f"{name}: {n_params:,} parameters, trained in {format_duration(seconds)} "
          f"({epochs} epochs)")

## Results

In [ ]:
header = f"{'Model':<13}{'Parameters':>12}{'Training time':>15}{'s / epoch':>11}{'Epochs':>8}{'Best val NLL':>14}{'Best val MSE':>14}"
print(header)
print("-" * len(header))
for name, r in results.items():
    print(f"{name:<13}{r['parameters']:>12,}{r['training_time']:>15}{r['seconds_per_epoch']:>11.2f}"
          f"{r['epochs_run']:>8}{r['best_val_nll']:>14.4f}{r['best_val_mse']:>14.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, h in histories.items():
    epochs = range(1, len(h["train_loss"]) + 1)
    axes[0].plot(epochs, h["train_loss"], label=name)
    axes[1].plot(h["val_epoch"], h["val_nll"], label=name)
    axes[1].plot(h["val_epoch"], h["val_elbo"], label=name)
    axes[2].plot(h["val_epoch"], h["val_mse"], label=name)
for ax, title in zip(axes, ["Training loss (negative ELBO)", "Validation NLL per value", "Validation MSE (standardized)"]):
    ax.axvline(EPOCHS, color="gray", ls=":", lw=1)           # phase 2 starts
    ax.axvline(KL_RAMP_EPOCHS, color="gray", ls="--", lw=1)  # KL ramp ends
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
axes[2].set_yscale("log")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

## Test set
Every mesh node at t0 = 30, for lags 0, 9 and 19, with the best weights of each model.
- **NLL** per node, of the predictive mixture over the latent samples (in standardized units).
- **RMSE** of the predictive mean, in physical units.

In [ ]:
@torch.no_grad()
def test_metrics(model, lag, n_samples=TEST_MC_SAMPLES):
    model.eval()
    nll_sum, se_sum, count = 0.0, 0.0, 0
    for i in range(len(data.y["test"])):  # one trajectory at a time: all nodes, many samples
        b = {k: v.to(device) for k, v in make_test_batch(data, [i], SENSORS, USE_MU, TEST_TIME, lag).items()}
        out = model(b["x_context"], b["y_context"], b["x_target"], num_samples=n_samples)
        log_p = out.y.log_prob(b["y_target"].unsqueeze(0)).sum(-1)           # (S, 1, nodes)
        nll_sum -= (torch.logsumexp(log_p, dim=0) - math.log(n_samples)).sum().item()
        mean = out.y.mean.mean(0) * data.y_std + data.y_mean                # physical units
        se_sum += ((mean - (b["y_target"] * data.y_std + data.y_mean)) ** 2).sum().item()
        count += b["y_target"].shape[1]
    return nll_sum / count, math.sqrt(se_sum / count)

torch.manual_seed(SEED)
print(f"{'Model':<13}" + "".join(f"{'lag ' + str(lag) + ' NLL':>14}{'RMSE':>10}" for lag in TEST_LAGS))
for name, model in models.items():
    results[name]["test"] = {}
    row = f"{name:<13}"
    for lag in TEST_LAGS:
        nll, rmse = test_metrics(model, lag)
        results[name]["test"][f"lag_{lag}"] = {"nll": nll, "rmse": rmse}
        row += f"{nll:>14.4f}{rmse:>10.4g}"
    print(row)
print(f"(std of y: {data.y_std:.4g})")
save_results()
print(f"\nSaved to {OUTPUT_DIR / 'results.json'}")

### Example: one test trajectory
Truth, predictive mean and predictive standard deviation at t0 = 30, lag 9.

In [ ]:
from dolfin import Function, plot as fenics_plot


def plot_field(ax, values, title, cmap="jet", vmin=None, vmax=None):
    field = Function(data.function_space)
    field.vector()[:] = values.numpy()
    plt.sca(ax)
    mappable = fenics_plot(field, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(mappable, ax=ax, shrink=0.75)
    ax.scatter(*data.coordinates[SENSORS].T, color="red", s=20)
    ax.set_title(title)
    ax.axis("off")


b = {k: v.to(device) for k, v in make_test_batch(data, [0], SENSORS, USE_MU, TEST_TIME, 9).items()}
truth = (b["y_target"][0, :, 0] * data.y_std + data.y_mean).cpu()
fig, axes = plt.subplots(len(models), 3, figsize=(18, 4 * len(models)), squeeze=False)
torch.manual_seed(SEED)
for row, (name, model) in enumerate(models.items()):
    with torch.no_grad():
        out = model(b["x_context"], b["y_context"], b["x_target"], num_samples=TEST_MC_SAMPLES)
    means = out.y.mean[:, 0, :, 0]                                            # (S, nodes)
    mean = (means.mean(0) * data.y_std + data.y_mean).cpu()
    std = ((out.y.variance[:, 0, :, 0].mean(0) + means.var(0, unbiased=False)).sqrt() * data.y_std).cpu()
    vmin, vmax = min(truth.min(), mean.min()).item(), max(truth.max(), mean.max()).item()
    plot_field(axes[row, 0], truth, "Truth", vmin=vmin, vmax=vmax)
    plot_field(axes[row, 1], mean, f"{name}: mean", vmin=vmin, vmax=vmax)
    plot_field(axes[row, 2], std, f"{name}: std", cmap="magma")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "test_example_t30_lag9.png", dpi=150)
plt.show()